# Exemple de génération de données PhysiNN

Ce notebook montre comment utiliser le script `examples/data_generation.py` pour charger les transitions spectrales depuis un fichier YAML, générer un jeu de données synthétique et visualiser les spectres ainsi que la distribution des paramètres physiques.

In [ ]:
from pathlib import Path

import torch

from examples.data_generation import (
    ConstantTips,
    NORMALISATION_PRESETS,
    NORMALIZATION,
    SpectraDataset,
    _build_transitions,
    _extract_poly_coeff,
    _load_normalisation,
    _load_spectra_config,
    _sample_dataset,
)
from examples.visualization import plot_param_hist, plot_spectra_example


In [ ]:
torch.manual_seed(123)

config_path = Path("examples/config/spectra_config.yaml")
spectra_config = _load_spectra_config(config_path)
transitions = _build_transitions(spectra_config)
poly_freq_ch4 = _extract_poly_coeff(spectra_config, "CH4")

normalisation = _load_normalisation("train_default", None)
NORMALIZATION.clear()
NORMALIZATION.update(normalisation)

dataset = SpectraDataset(
    n_samples=64,
    num_points=800,
    poly_freq_CH4=poly_freq_ch4,
    transitions_dict=transitions,
    sample_ranges=normalisation,
    strict_check=True,
    with_noise=False,
    noise_profile=None,
    freeze_noise=False,
    tipspy=ConstantTips(),
)

spectra_df, params_df = _sample_dataset(dataset, n_samples=8)
spectra_df.head()


In [ ]:
fig_spectra = plot_spectra_example(spectra_df)
fig_params = plot_param_hist(params_df)
fig_spectra
fig_params
